In [1]:
import torch, torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM

config = dict(decoder = "gpt2", prefix = "patches", epochs = 2, batch_size = 4, accum_grad = 8, lr = 5e-5, max_len = 950, lora = False)
device = 'cuda'
device

'cuda'

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!cp /content/drive/MyDrive/cadquery/*.pt /content/

In [4]:
tokenizer = AutoTokenizer.from_pretrained(config["decoder"])

if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

In [5]:
class data(Dataset):
  def __init__(self, path):
    data = torch.load(path)
    self.f, self.c = data["feats"], data["codes"]

  def __len__(self):
    return len(self.c)

  def __getitem__(self, i):
    input_ids = tokenizer(self.c[i] + tokenizer.eos_token, truncation = True, max_length = config["max_len"])["input_ids"]
    return self.f[i], torch.tensor(input_ids)

In [6]:
def collate(batch):
  features = torch.stack([f[0] for f in batch])
  if config["prefix"] == "cls":
    features = features[:,:1]
  token_ids = [c[1] for c in batch]
  L = max(len(i) for i in token_ids)
  pad = torch.full((len(batch), L), tokenizer.pad_token_id)
  labels = torch.full((len(batch), L), -100)
  for row, ids in enumerate(token_ids):
      pad[row, :len(ids)] = ids
      labels[row, :len(ids)] = ids
  return features, pad, labels

In [8]:
class Model(nn.Module):
  def __init__(self):
    super().__init__()
    self.decoder = AutoModelForCausalLM.from_pretrained(config["decoder"], torch_dtype = torch.float32).to(device)
    hidden_size = self.decoder.config.hidden_size
    self.proj = nn.Sequential(nn.Linear(768, hidden_size), nn.GELU(), nn.Linear(hidden_size, hidden_size)).to(device)

  def forward(self, feats, ids, labels):
      pre = self.proj(feats.to(device, torch.float32))          # [B,P,H]
      emb = self.decoder.get_input_embeddings()(ids.to(device))     # [B,L,H]
      x = torch.cat([pre, emb], 1)
      P = pre.shape[1]
      lab = torch.cat([torch.full((ids.size(0), P), -100, device=device),
                        labels.to(device)], 1)
      mask = torch.ones(x.shape[:2], device=device)
      mask[:, P:] = (ids.to(device) != tokenizer.pad_token_id).long()
      return self.decoder(inputs_embeds=x, attention_mask=mask, labels=lab).loss

In [9]:
eval_dl = DataLoader(data("/content/eval.pt"), batch_size=8, collate_fn=collate)

@torch.no_grad()
def val_loss():
    m.eval()
    tot = 0
    for f, i, l in eval_dl:
        with torch.autocast("cuda", torch.float16):
            tot += m(f, i, l).item()
    m.train()
    return tot / len(eval_dl)

In [10]:
m = Model()
dl = DataLoader(data("/content/train.pt"), batch_size = config["batch_size"], shuffle = True, collate_fn = collate)

optimizer = torch.optim.AdamW(m.parameters(), lr = config["lr"])
scaler = torch.amp.GradScaler()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [11]:
for ep in range(config["epochs"]):
  for step, (f, i, l) in enumerate(dl):
    with torch.autocast("cuda", torch.float16):
        loss = m(f, i, l) / config["accum_grad"]
    scaler.scale(loss).backward()
    if (step + 1) % config["accum_grad"] == 0:
        scaler.step(optimizer); scaler.update(); optimizer.zero_grad()
    if step % 100 == 0:
        print(ep, step, loss.item() * config["accum_grad"])
  print(f"epoch {ep} val_loss {val_loss():.4f}")
torch.save({"proj": m.proj.state_dict(), "dec": m.decoder.state_dict()}, "/content/run_b.pt")

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


0 0 3.108724355697632
0 100 0.5717046856880188
0 200 0.2858128845691681
0 300 0.3523081839084625
0 400 0.47434547543525696
0 500 0.2773762345314026
0 600 0.31752538681030273
0 700 0.14060968160629272
0 800 0.3336430490016937
0 900 0.1938255876302719
0 1000 0.17413145303726196
0 1100 0.25689682364463806
0 1200 0.5493459701538086
0 1300 0.33794981241226196
0 1400 0.1513529270887375
0 1500 0.46821391582489014
0 1600 0.2942822277545929
0 1700 0.5660030245780945
0 1800 0.15721960365772247
0 1900 0.47969162464141846
0 2000 0.2369721382856369
0 2100 0.19600282609462738
0 2200 0.25976574420928955
0 2300 0.14831425249576569
0 2400 0.18346430361270905
epoch 0 val_loss 0.2485
1 0 0.9196957945823669
1 100 0.19863517582416534
1 200 0.23657046258449554
1 300 0.28539708256721497
1 400 0.1966555416584015
1 500 0.2347826510667801
1 600 0.47234317660331726
1 700 0.18350964784622192
1 800 0.1536354273557663
1 900 0.26250702142715454
1 1000 0.20185533165931702
1 1100 0.16887375712394714
1 1200 0.124363794

In [15]:
m = Model()
ck = torch.load("run_a.pt")
m.proj.load_state_dict(ck["proj"])
m.decoder.load_state_dict(ck["dec"])
m.eval()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Model(
  (decoder): GPT2LMHeadModel(
    (transformer): GPT2Model(
      (wte): Embedding(50257, 768)
      (wpe): Embedding(1024, 768)
      (drop): Dropout(p=0.1, inplace=False)
      (h): ModuleList(
        (0-11): 12 x GPT2Block(
          (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (attn): GPT2Attention(
            (c_attn): Conv1D(nf=2304, nx=768)
            (c_proj): Conv1D(nf=768, nx=768)
            (attn_dropout): Dropout(p=0.1, inplace=False)
            (resid_dropout): Dropout(p=0.1, inplace=False)
          )
          (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): GPT2MLP(
            (c_fc): Conv1D(nf=3072, nx=768)
            (c_proj): Conv1D(nf=768, nx=3072)
            (act): NewGELUActivation()
            (dropout): Dropout(p=0.1, inplace=False)
          )
        )
      )
      (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    )
    (lm_head): Linear(in_features=768, out_features=5025

In [16]:
# torch.save({"proj": m.proj.state_dict(), "dec": m.decoder.state_dict()}, "/content/run_a.pt")

d = torch.load("/content/eval.pt")
feats = d["feats"]

m.eval()
preds = []
with torch.no_grad():
    for i in range(100):
        pre = m.proj(feats[i:i+1, :1].cuda().float())
        out = m.decoder.generate(inputs_embeds=pre, max_new_tokens=512, pad_token_id=tokenizer.eos_token_id, eos_token_id=tokenizer.eos_token_id)
        preds.append(tokenizer.decode(out[0], skip_special_tokens=True))

torch.save({"preds": preds, "gt": d["codes"][:100]}, "/content/preds_a_again.pt")
print(preds[0])

import cadquery as cq
# Generating a workplane for sketch 0
wp_sketch0 = cq.Workplane(cq.Plane(cq.Vector(-0.75, 0.0, 0.0), cq.Vector(1.0, 6.123233995736766e-17, -6.123233995736766e-17), cq.Vector(6.123233995736766e-17, -1.0, 6.123233995736766e-17)))
loop0=wp_sketch0.moveTo(0.7578947368421053, 0.0).lineTo(0.7578947368421053, 0.7578947368421053).lineTo(0.0, 0.7578947368421053).lineTo(0.0, 0.0).close()
solid0=wp_sketch0.add(loop0).extrude(0.75)
solid=solid0
# Generating a workplane for sketch 1
wp_sketch1 = cq.Workplane(cq.Plane(cq.Vector(-0.75, 0.0, 0.0), cq.Vector(1.0, 6.123233995736766e-17, -6.123233995736766e-17), cq.Vector(6.123233995736766e-17, -1.0, 6.123233995736766e-17)))
loop1=wp_sketch1.moveTo(0.7578947368421053, 0.0).lineTo(0.7578947368421053, 0.7578947368421053).lineTo(0.0, 0.7578947368421053).lineTo(0.0, 0.0).close()
solid1=wp_sketch1.add(loop1).extrude(-0.75)
solid=solid.cut(solid1)
# Generating a workplane for sketch 2
wp_sketch2 = cq.Workplane(cq.Plane(cq.Vector(-0.75, 0.